# Eksperimen Preprocessing Data - Credit Scoring
**Nama**: Ragil Amirzaky  
**Email**: d1041241069@student.untan.ac.id  
**Kelas**: Membangun Sistem Machine Learning (Dicoding)  

Notebook ini berisi alur eksperimen eksplorasi data (EDA) dan preprocessing manual sebelum dikonversi menjadi skrip otomatisasi.

## 1. Import Library

In [1]:
import pandas as pd
import numpy as np
import os

print("Library berhasil di-import!")

## 2. Data Loading

In [2]:
# Path dataset raw
raw_path = '../credit_scoring_raw/cs-training.csv'
if not os.path.exists(raw_path):
    raw_path = 'credit_scoring_raw/cs-training.csv'

df = pd.read_csv(raw_path)
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

print(f"Ukuran dataset raw: {df.shape}")
df.head()

## 3. Exploratory Data Analysis (EDA)

In [3]:
# Ringkasan Informasi Data
print("--- Informasi Dataset ---")
print(df.info())

print("\n--- Statistik Deskriptif ---")
display(df.describe())

print("\n--- Distribusi Target (SeriousDlqin2yrs) ---")
print(df['SeriousDlqin2yrs'].value_counts(normalize=True))

print("\n--- Jumlah Missing Values ---")
print(df.isnull().sum())

## 4. Data Preprocessing & Cleaning

In [4]:
# 4.1 Handling Missing Values
df_clean = df.copy()

# Imputasi Median untuk MonthlyIncome
median_income = df_clean['MonthlyIncome'].median()
df_clean['MonthlyIncome'] = df_clean['MonthlyIncome'].fillna(median_income)

# Imputasi Modus untuk NumberOfDependents
mode_dependents = df_clean['NumberOfDependents'].mode()[0]
df_clean['NumberOfDependents'] = df_clean['NumberOfDependents'].fillna(mode_dependents)

print("Missing values setelah imputasi:")
print(df_clean.isnull().sum())

In [5]:
# 4.2 Outlier Handling (Metode IQR)
outlier_cols = ['RevolvingUtilizationOfUnsecuredLines', 'DebtRatio']
for col in outlier_cols:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    upper_bound = Q3 + 3 * IQR
    lower_bound = Q1 - 3 * IQR
    df_clean[col] = np.clip(df_clean[col], lower_bound, upper_bound)

print("Pembersihan outlier selesai!")

In [6]:
# 4.3 Feature Engineering
df_clean['TotalLatePayments'] = (
    df_clean['NumberOfTime30-59DaysPastDueNotWorse'] +
    df_clean['NumberOfTime60-89DaysPastDueNotWorse'] +
    df_clean['NumberOfTimes90DaysLate']
)
df_clean['IncomePerDependent'] = df_clean['MonthlyIncome'] / (df_clean['NumberOfDependents'] + 1)

print("Fitur baru berhasil ditambahkan:")
print(df_clean[['TotalLatePayments', 'IncomePerDependent']].head())

## 5. Data Preprocessing & Export

In [7]:
output_dir = 'preprocessing'
os.makedirs(output_dir, exist_ok=True)
output_file = os.path.join(output_dir, 'credit_scoring_preprocessing.csv')

df_clean.to_csv(output_file, index=False)
print(f"Data preprocessed berhasil disimpan ke {output_file}")